In [ ]:
import pandas as pd

# Load the files
imdb_df = pd.read_csv('IMDB top 1000.csv')
netflix_df = pd.read_csv('netflix_titles.csv')

# Column names
print("IMDb Columns:", imdb_df.columns)
print("Netflix Columns:", netflix_df.columns)

In [ ]:

# Title conflict resolution:
# IMDb stores titles as "1. The Shawshank Redemption (1994)"
# Netflix stores titles as "The Shawshank Redemption"
# We strip the ranking prefix and year suffix from IMDb titles to enable matching
imdb_df['clean_title'] = imdb_df['Title'].str.replace(r'^\d+\.\s+', '', regex=True) # Βγάζει το "1. "
imdb_df['clean_title'] = imdb_df['clean_title'].str.replace(r'\s\(\d{4}\)$', '', regex=True) # Βγάζει το "(1994)"
imdb_df['clean_title'] = imdb_df['clean_title'].str.lower().str.strip()

netflix_df['clean_title'] = netflix_df['title'].str.lower().str.strip()

# Filter Netflix to movies only before merging (excludes TV shows)
merged_df = pd.merge(imdb_df, netflix_df[netflix_df['type'] == 'Movie'], on='clean_title', how='inner')

print(f"Found {len(merged_df)} movies in common.")

In [ ]:
# Temporal Bias Analysis
# We extract the release decade from each dataset to examine
# whether Netflix favors more recent films over classic ones

imdb_df['year'] = imdb_df['Title'].str.extract(r'\((\d{4})\)').astype(float)
imdb_df['decade'] = (imdb_df['year'] // 10 * 10).astype(int).astype(str) + 's'
merged_df['decade'] = (merged_df['release_year'] // 10 * 10).astype(int).astype(str) + 's'

total_distribution = imdb_df['decade'].value_counts().sort_index()
netflix_distribution = merged_df['decade'].value_counts().sort_index()

comparison_table = pd.DataFrame({
    'IMDb Top 1000': total_distribution,
    'Found on Netflix': netflix_distribution
}).fillna(0).astype(int)

comparison_table['Coverage %'] = (comparison_table['Found on Netflix'] / comparison_table['IMDb Top 1000'] * 100).round(1)

print(comparison_table)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# 1. Prepare & Sort Data (Ensure decades are in chronological order)
sorted_decades = sorted(comparison_table.index, key=lambda x: int(x[:-1]))
comparison_table.index = pd.Categorical(comparison_table.index, categories=sorted_decades, ordered=True)
df_plot = comparison_table.sort_index()

# 2. Convert to numpy arrays for clean plotting
labels = df_plot.index.astype(str)
imdb_vals = df_plot['IMDb Top 1000'].values
netflix_vals = df_plot['Found on Netflix'].values
coverage_vals = df_plot['Coverage %'].values

x = np.arange(len(labels))
width = 0.4

fig, ax1 = plt.subplots(figsize=(14, 8))

# 3. Create Grouped Bars
ax1.bar(x - width/2, imdb_vals, width, label='IMDb Top 1000', color='#3498db', alpha=0.7)
ax1.bar(x + width/2, netflix_vals, width, label='Found on Netflix', color='#e74c3c', alpha=0.9)

# Axis 1 Settings (Quantities)
ax1.set_xlabel('Decade', fontsize=12, fontweight='bold')
ax1.set_ylabel('Number of Movies', fontsize=12, fontweight='bold')
ax1.set_title('Temporal Bias Analysis: IMDb vs Netflix Coverage', fontsize=16, pad=20, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(labels, rotation=45)
ax1.legend(loc='upper left')
ax1.grid(axis='y', linestyle='--', alpha=0.3)

# 4. Axis 2 for Coverage % (Line Chart)
ax2 = ax1.twinx()
ax2.plot(x, coverage_vals, color='#27ae60', marker='D', markersize=8, linewidth=3, label='Coverage %')
ax2.set_ylabel('Coverage Percentage (%)', color='#27ae60', fontsize=12, fontweight='bold')
ax2.set_ylim(0, 100)

# Add percentage labels above points
for i, val in enumerate(coverage_vals):
    ax2.annotate(f'{val}%', (x[i], val), xytext=(0, 10),
                 textcoords='offset points', ha='center', color='#27ae60', fontweight='bold')

fig.tight_layout()
plt.show()

In [ ]:
# Genre Distribution of shared movies using Netflix labels
# This shows which Netflix genre categories the matched IMDb films fall into
genre_series = merged_df['listed_in'].dropna().str.split(',').explode().str.strip()

genre_counts = genre_series.value_counts()

# Plot
fig, ax = plt.subplots(figsize=(14, 7))
bars = ax.barh(genre_counts.index[:15], genre_counts.values[:15], color='#e74c3c', alpha=0.85)

# Add value labels
for bar, val in zip(bars, genre_counts.values[:15]):
    ax.text(val + 0.3, bar.get_y() + bar.get_height() / 2,
            str(val), va='center', fontweight='bold')

ax.set_xlabel('Number of Movies', fontsize=12, fontweight='bold')
ax.set_title('Top 15 Genres: IMDb Top 1000 Movies Found on Netflix', fontsize=14, fontweight='bold', pad=15)
ax.invert_yaxis()
ax.grid(axis='x', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ---- DURATION & RUNTIME TRENDS ----

# Netflix stores duration as "120 min" strings — extract numeric value
merged_df['runtime_min'] = merged_df['duration'].str.extract(r'(\d+)').astype(float)

# IMDb may have its own runtime column — check and use it if available
# (adjust column name if yours differs, e.g. 'Runtime', 'runtime')
if 'Runtime' in merged_df.columns:
    imdb_df['runtime_clean'] = imdb_df['Runtime'].str.extract(r'(\d+)').astype(float)

# --- Plot 1: Runtime Distribution (Histogram) ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].hist(merged_df['runtime_min'].dropna(), bins=20, color='#3498db', alpha=0.8, edgecolor='white')
axes[0].axvline(merged_df['runtime_min'].mean(), color='#e74c3c', linestyle='--', linewidth=2,
                label=f"Mean: {merged_df['runtime_min'].mean():.0f} min")
axes[0].set_xlabel('Runtime (minutes)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Count', fontsize=12, fontweight='bold')
axes[0].set_title('Runtime Distribution\n(IMDb Top 1000 on Netflix)', fontsize=13, fontweight='bold')
axes[0].legend()
axes[0].grid(axis='y', linestyle='--', alpha=0.3)

# --- Plot 2: Avg Runtime per Decade ---
avg_runtime_by_decade = merged_df.groupby('decade')['runtime_min'].mean().sort_index()

axes[1].plot(avg_runtime_by_decade.index, avg_runtime_by_decade.values,
             marker='o', color='#27ae60', linewidth=3, markersize=8)

for i, (dec, val) in enumerate(avg_runtime_by_decade.items()):
    axes[1].annotate(f'{val:.0f}m', (dec, val), xytext=(0, 10),
                     textcoords='offset points', ha='center', fontweight='bold', color='#27ae60')

axes[1].set_xlabel('Decade', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Avg Runtime (min)', fontsize=12, fontweight='bold')
axes[1].set_title('Average Movie Runtime by Decade\n(Shared IMDb/Netflix Movies)', fontsize=13, fontweight='bold')
axes[1].grid(linestyle='--', alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Genre Bias Analysis
# IMDb and Netflix use entirely different genre taxonomies — this is a schema-level conflict.
# IMDb uses precise cinematic categories (Biography, Crime, Mystery, War, History)
# Netflix uses broad mood-based labels (Thrillers, Action & Adventure, Sci-Fi & Fantasy)
# We build a manual crosswalk to map Netflix labels to their IMDb equivalents
# Labels like 'International Movies', 'Classic Movies' describe origin/era, not genre, so they are dropped
netflix_to_imdb = {
    'Dramas': ['Drama'],
    'International Movies': [],  # origin, not genre — drop
    'Action & Adventure': ['Action', 'Adventure'],
    'Independent Movies': [],  # style, not genre — drop
    'Comedies': ['Comedy'],
    'Thrillers': ['Thriller', 'Crime', 'Mystery'],  # Netflix bundles these into Thrillers
    'Romantic Movies': ['Romance'],
    'Sci-Fi & Fantasy': ['Sci-Fi', 'Fantasy'],
    'Classic Movies': [],  # era, not genre — drop
    'LGBTQ Movies': [],  # identity label, not genre — drop
    'Cult Movies': [],  # style, not genre — drop
    'Children & Family Movies': ['Family', 'Animation'],  # kids content = animated too
    'Sports Movies': ['Sport'],
    'Music & Musicals': ['Music'],
    'Horror Movies': ['Horror'],
}

# Genres with no Netflix equivalent — a finding in itself
print("""
NETFLIX GENRE BLIND SPOTS (no label exists at all):
- Biography  → Netflix doesn't categorize biopics separately
- History    → Absorbed into Dramas or International Movies
- War        → Absorbed into Action & Adventure or Dramas
- Animation  → Only 'Children & Family' and 'Anime Features' exist
- Crime      → Bundled inside Thrillers
- Mystery    → Bundled inside Thrillers
""")


# Now each Netflix label maps to a LIST of IMDb genres
def normalize_netflix_genres(genre_str):
    if pd.isna(genre_str):
        return []
    result = []
    for g in genre_str.split(','):
        g = g.strip()
        result.extend(netflix_to_imdb.get(g, []))
    return result


merged_df['normalized_genres'] = merged_df['listed_in'].apply(normalize_netflix_genres)

imdb_genres = imdb_df['Genre'].dropna().str.split(',').explode().str.strip()
imdb_genre_counts = imdb_genres.value_counts()


netflix_normalized = merged_df.explode('normalized_genres')
netflix_genre_counts = netflix_normalized['normalized_genres'].dropna().value_counts()


genre_comparison = pd.DataFrame({
    'IMDb Top 1000': imdb_genre_counts,
    'On Netflix': netflix_genre_counts
}).fillna(0).astype(int)

genre_comparison['Coverage %'] = (
        genre_comparison['On Netflix'] / genre_comparison['IMDb Top 1000'] * 100
).round(1).clip(upper=100)

genre_comparison = genre_comparison.sort_values('IMDb Top 1000', ascending=False).head(15)

print(genre_comparison)

# --- Same plot as before, now with real data ---
fig, ax1 = plt.subplots(figsize=(16, 8))

x = np.arange(len(genre_comparison))
width = 0.4

ax1.bar(x - width / 2, genre_comparison['IMDb Top 1000'], width,
        label='IMDb Top 1000', color='#3498db', alpha=0.8)
ax1.bar(x + width / 2, genre_comparison['On Netflix'], width,
        label='On Netflix', color='#e74c3c', alpha=0.85)

ax1.set_xlabel('Genre', fontsize=12, fontweight='bold')
ax1.set_ylabel('Number of Movies', fontsize=12, fontweight='bold')
ax1.set_title('Genre Bias: IMDb Top 1000 vs Netflix Coverage', fontsize=15, fontweight='bold', pad=20)
ax1.set_xticks(x)
ax1.set_xticklabels(genre_comparison.index, rotation=40, ha='right')
ax1.legend(loc='upper right')
ax1.grid(axis='y', linestyle='--', alpha=0.3)

ax2 = ax1.twinx()
ax2.plot(x, genre_comparison['Coverage %'], color='#27ae60',
         marker='D', markersize=8, linewidth=2.5, label='Coverage %')
ax2.set_ylabel('Coverage %', color='#27ae60', fontsize=12, fontweight='bold')
ax2.set_ylim(0, 110)

for i, val in enumerate(genre_comparison['Coverage %']):
    ax2.annotate(f'{val}%', (x[i], val), xytext=(0, 10),
                 textcoords='offset points', ha='center',
                 color='#27ae60', fontweight='bold', fontsize=9)

ax2.legend(loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# See ALL unique Netflix genre labels in the shared movies
all_netflix_labels = merged_df['listed_in'].dropna().str.split(',').explode().str.strip()
print(all_netflix_labels.value_counts().to_string())

In [ ]:
# Age Certification Bias Analysis
# IMDb mixes rating systems from multiple countries (UK, US, India etc.)
# Netflix uses its own proprietary rating taxonomy (TV-MA, TV-14, TV-PG)
# This is a schema-level conflict — the same restriction level has different labels per source
# We normalize both to a common 5-tier system: G, PG, PG-13, R, NC-17, Not Rated
imdb_cert_map = {
    'G': 'G', 'U': 'G',  # Universal/General
    'PG': 'PG', 'U/A': 'PG', 'GP': 'PG',  # Parental Guidance
    'PG-13': 'PG-13', '12A': 'PG-13', '12': 'PG-13',  # Teen
    'R': 'R', '15': 'R', '18': 'R', 'A': 'R',  # Restricted/Adult
    'NC-17': 'NC-17', 'X': 'NC-17',  # Adults Only
    'Not Rated': 'Not Rated', 'Unrated': 'Not Rated', 'Passed': 'Not Rated'
}

# Normalize Netflix ratings to same categories
netflix_cert_map = {
    'G': 'G',
    'PG': 'PG',
    'PG-13': 'PG-13', 'TV-14': 'PG-13',
    'R': 'R', 'TV-MA': 'R',
    'NC-17': 'NC-17',
    'NR': 'Not Rated', 'UR': 'Not Rated', 'TV-G': 'G', 'TV-PG': 'PG'
}

# Apply mappings
imdb_df['cert_normalized'] = imdb_df['Certificate'].map(imdb_cert_map).fillna('Not Rated')
merged_df['cert_normalized'] = merged_df['rating'].map(netflix_cert_map).fillna('Not Rated')

# Define consistent order
cert_order = ['G', 'PG', 'PG-13', 'R', 'NC-17', 'Not Rated']
colors = ['#2ecc71', '#3498db', '#f39c12', '#e74c3c', '#8e44ad', '#95a5a6']

# Count distributions
imdb_cert_counts = imdb_df['cert_normalized'].value_counts().reindex(cert_order, fill_value=0)
netflix_cert_counts = merged_df['cert_normalized'].value_counts().reindex(cert_order, fill_value=0)

# Normalize to percentages for fair comparison (different total sizes)
imdb_cert_pct = (imdb_cert_counts / imdb_cert_counts.sum() * 100).round(1)
netflix_cert_pct = (netflix_cert_counts / netflix_cert_counts.sum() * 100).round(1)

fig, axes = plt.subplots(1, 3, figsize=(20, 7))

# --- Plot 1: IMDb Certificate Distribution (Pie) ---
axes[0].pie(imdb_cert_counts, labels=cert_order, colors=colors,
            autopct='%1.1f%%', startangle=140, pctdistance=0.8)
axes[0].set_title('IMDb Top 1000\nCertificate Distribution', fontsize=13, fontweight='bold')

# --- Plot 2: Netflix Certificate Distribution (Pie) ---
axes[1].pie(netflix_cert_counts, labels=cert_order, colors=colors,
            autopct='%1.1f%%', startangle=140, pctdistance=0.8)
axes[1].set_title('Netflix (Shared Movies)\nCertificate Distribution', fontsize=13, fontweight='bold')

# --- Plot 3: Side-by-side % comparison bar chart ---
x = np.arange(len(cert_order))
width = 0.35

bars1 = axes[2].bar(x - width / 2, imdb_cert_pct, width,
                    label='IMDb Top 1000', color='#3498db', alpha=0.85)
bars2 = axes[2].bar(x + width / 2, netflix_cert_pct, width,
                    label='On Netflix', color='#e74c3c', alpha=0.85)

# Value labels on bars
for bar in bars1:
    h = bar.get_height()
    if h > 0:
        axes[2].text(bar.get_x() + bar.get_width() / 2, h + 0.5,
                     f'{h}%', ha='center', fontsize=9, fontweight='bold', color='#3498db')

for bar in bars2:
    h = bar.get_height()
    if h > 0:
        axes[2].text(bar.get_x() + bar.get_width() / 2, h + 0.5,
                     f'{h}%', ha='center', fontsize=9, fontweight='bold', color='#e74c3c')

axes[2].set_xticks(x)
axes[2].set_xticklabels(cert_order)
axes[2].set_ylabel('Percentage (%)', fontsize=12, fontweight='bold')
axes[2].set_title('Certificate Distribution Comparison\n(% of total)', fontsize=13, fontweight='bold')
axes[2].legend()
axes[2].grid(axis='y', linestyle='--', alpha=0.3)

plt.suptitle('Age Rating Bias: IMDb Top 1000 vs Netflix Coverage',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

SECTION 3- IDENTIFYING BIAS USING FAIRLEARN

In [ ]:
# 1. Install fairlearn if you haven't yet (Run in terminal)
# !pip install fairlearn

import pandas as pd
from fairlearn.metrics import MetricFrame
import numpy as np

# Prepare the data
# We'll create a simplified 'primary_country' feature from Netflix
merged_df['primary_country'] = merged_df['country'].fillna('Unknown').apply(lambda x: x.split(',')[0])

# We'll filter for countries with at least 5 movies to make the results meaningful
top_countries = merged_df['primary_country'].value_counts()[merged_df['primary_country'].value_counts() >= 5].index
df_fairness = merged_df[merged_df['primary_country'].isin(top_countries)].copy()

print(f"Analyzing fairness across top countries (at least 5 movies each): {list(top_countries)}")

In [ ]:
import pandas as pd
import numpy as np
from fairlearn.metrics import MetricFrame, selection_rate
import matplotlib.pyplot as plt

# 1. AGGRESSIVE CLEANING
# Ensure numeric, no NaNs, and reset index
df_fair = merged_df.copy()
df_fair['Rate'] = pd.to_numeric(df_fair['Rate'], errors='coerce')
df_fair['primary_country'] = df_fair['country'].fillna('Unknown').apply(lambda x: str(x).split(',')[0].strip())
df_fair = df_fair.dropna(subset=['Rate', 'primary_country'])

# Filter for countries with at least 5 movies for statistical validity
counts = df_fair['primary_country'].value_counts()
top_countries = counts[counts >= 5].index
df_fair = df_fair[df_fair['primary_country'].isin(top_countries)].copy()

# Create binary label for "High Rated" movies
df_fair['is_high_rated'] = (df_fair['Rate'] >= 8.0).astype(int)

# 2. CONVERT TO LISTS (This avoids the "Scalar Index" Error)
# By using tolist(), we remove all Pandas/Numpy indexing metadata
y_true_list = df_fair['Rate'].tolist()
sf_list = df_fair['primary_country'].tolist()
binary_y_list = df_fair['is_high_rated'].tolist()

print(f"Data ready. Analyzing {len(df_fair)} movies across {len(top_countries)} countries.\n")

try:
    # 3. FAIRLEARN ANALYSIS
    metrics_dict = {'avg_rating': np.mean}

    mf = MetricFrame(
        metrics=metrics_dict,
        y_true=y_true_list,
        y_pred=y_true_list, # We use y_true as y_pred to get descriptive stats by group
        sensitive_features=sf_list
    )

    print("--- Bias Identification (Fairlearn) ---")
    print(mf.by_group)
    print(f"\nMax Rating Gap: {mf.difference()['avg_rating']:.2f}")

except Exception as e:
    print(f"Fairlearn still having issues: {e}")
    print("\n--- Plan B: Identifying Bias with Pandas (Manual Calculation) ---")
    # This is a valid "Tool" approach for the assignment!
    bias_analysis = df_fair.groupby('primary_country')['Rate'].mean().sort_values(ascending=False)
    print(bias_analysis)

    gap = bias_analysis.max() - bias_analysis.min()
    print(f"\nManual Bias Gap Calculation: {gap:.2f}")

# 4. VISUALIZATION OF BIAS
plt.figure(figsize=(10, 6))
df_fair.groupby('primary_country')['Rate'].mean().sort_values().plot(kind='barh', color='skyblue')
plt.title('Average IMDb Rating by Production Country')
plt.xlabel('IMDb Rate')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.show()

The Disparate impact ratio

In [ ]:
from fairlearn.metrics import demographic_parity_ratio

# We calculate the ratio between the worst-off country and the best-off country
# This uses the binary 'is_high_rated' (Rate >= 8.0) we created earlier
dp_ratio = demographic_parity_ratio(
    y_true=df_fair['is_high_rated'].tolist(),
    y_pred=df_fair['is_high_rated'].tolist(),
    sensitive_features=df_fair['primary_country'].tolist()
)

print(f"Demographic Parity Ratio: {dp_ratio:.4f}")

if dp_ratio < 0.80:
    print(f"Conclusion: Disparate Impact detected ({dp_ratio:.2f} < 0.80). The integration favors certain countries.")
else:
    print("Conclusion: The dataset is relatively balanced according to the 80% rule.")

Intersectional Bias (Data sparsity between the top 1000 and what is available in netflix)

In [ ]:
# 1. Create a single combined feature string
# --- ANALYSIS 2: Intersectional Bias (Manual) ---
# Goal: Identify bias at the intersection of Country + Genre
# We group by both columns and calculate the average rating
intersectional_bias = df_fair.groupby(['primary_country', 'main_genre'])['Rate'].mean().unstack()

print("\n--- Analysis 2: Intersectional Rating Table ---")
print(intersectional_bias)

# Visualization for Intersectional (Top 10 combinations)
plt.figure(figsize=(10, 5))
df_fair.groupby(['primary_country', 'main_genre'])['Rate'].mean().sort_values().tail(10).plot(kind='barh', color='salmon')
plt.title('Top 10 Country & Genre Combinations by Rating')
plt.show()


# Get all unique countries from the original Netflix dataset
# We take the first country listed if there are multiple
all_netflix_countries = set(netflix_df['country'].dropna().apply(lambda x: str(x).split(',')[0].strip()).unique())

# Get the countries that actually survived the merge
surviving_countries = set(df_fair['primary_country'].unique())

# Identify the missing ones
lost_countries = all_netflix_countries - surviving_countries

print(f"--- Data Integration Conflict: Representation Loss ---")
print(f"Original countries in Netflix: {len(all_netflix_countries)}")
print(f"Countries remaining after IMDb merge: {len(surviving_countries)}")
print(f"Countries lost: {len(lost_countries)}")

# Show a sample of lost countries
print("\nSample of countries with 0 representation in the merged dataset:")
print(list(lost_countries)[:15])

Temporal Bias

In [ ]:
# --- ANALYSIS 3: Temporal Bias (Classic vs Modern) ---

# STEP 1: Create the 'era' column explicitly
# We define 'Modern' as 2010 onwards, and 'Classic' as anything before
df_fair['era'] = df_fair['release_year'].apply(lambda x: 'Modern' if x >= 2010 else 'Classic')

# STEP 2: Group and calculate
era_analysis = df_fair.groupby('era')['Rate'].agg(['mean', 'count'])

print("--- Analysis 3: Temporal Bias Results ---")
print(era_analysis)

# STEP 3: Calculate the Gap
# Note: we use .loc to safely access the mean values
if 'Classic' in era_analysis.index and 'Modern' in era_analysis.index:
    classic_avg = era_analysis.loc['Classic', 'mean']
    modern_avg = era_analysis.loc['Modern', 'mean']
    era_gap = classic_avg - modern_avg
    print(f"\nThe 'Era Gap' is {era_gap:.2f} rating points.")
    print(f"Insight: {'Classic' if era_gap > 0 else 'Modern'} movies are rated higher in this integrated set.")

Integration selection bias
When merging the 2 datasets, did we keep the balance of representation among genres, or did we filter out some of them?

In [ ]:
# --- ANALYSIS 4: Integration Coverage Bias (Genre Filter) ---

# 1. Get genre counts from the original IMDb Top 1000 (The "Gold Standard")
# We use .str.split().explode() because movies have multiple genres (e.g. "Action, Drama")
imdb_genre_counts = imdb_df['Genre'].str.split(',').explode().str.strip().value_counts()

# 2. Get genre counts from your Merged Dataset (The "Integrated Result")
# We use the 'Genre' column from the IMDb side of the merge for a fair comparison
merged_genre_counts = df_fair['Genre'].str.split(',').explode().str.strip().value_counts()

# 3. Calculate Selection Rate (%)
# (How many of the Top 1000 movies of each genre actually made it into Netflix?)
genre_selection_rate = (merged_genre_counts / imdb_genre_counts).fillna(0) * 100

print("--- Analysis 4: Integration Selection Rate by Genre ---")
print("What % of each 'Top 1000' Genre is available on Netflix?")
print(genre_selection_rate.sort_values(ascending=False).round(2).to_string())

# 4. Visualization
plt.figure(figsize=(12, 6))
genre_selection_rate.sort_values().plot(kind='barh', color='mediumpurple')
plt.title('Integration Coverage: What % of IMDb Top 1000 Genres are on Netflix?')
plt.xlabel('Selection Rate (%)')
plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.show()